# 🚦 AUTOMATED Traffic Violation Model Training

This notebook automatically downloads a public dataset and trains your model!

**Dataset**: Kaggle - Smart Helmet Detection (YOLOv8 format)
- 🎯 Helmet / No-Helmet detection
- 📦 Ready-to-use annotations
- 📊 Train/Valid/Test split included

**Just run all cells!** ⚡

---

## Step 1: Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

# Install dependencies
!pip install -q ultralytics opendatasets kaggle

from IPython import display
display.clear_output()

print("✅ Environment ready!")
print("🎮 GPU:", "Available" if torch.cuda.is_available() else "Not available (CPU mode)")

import torch
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"⚡ CUDA available: {torch.cuda.is_available()}")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create directories
output_dir = '/content/drive/MyDrive/traffic-violation-model'
os.makedirs(output_dir, exist_ok=True)

print(f"✅ Google Drive mounted!")
print(f"📁 Model will save to: {output_dir}")

## Step 3: Download Dataset (Automatic)

### Option A: Download from Kaggle (RECOMMENDED)

In [ ]:
# Upload your Kaggle API credentials file (kaggle.json)
# Get it from: https://www.kaggle.com/settings → API → Create New Token

from google.colab import files
import os

print("📤 Upload your kaggle.json file (from Kaggle → Settings → API)")
uploaded = files.upload()

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ Kaggle credentials configured!")

In [ ]:
# Download Smart Helmet Detection Dataset
!kaggle datasets download -d aditya4567876543456/smart-helmet-detection-using-yolo-v8

# Extract dataset
import zipfile

with zipfile.ZipFile('smart-helmet-detection-using-yolo-v8.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/helmet-dataset')

print("✅ Dataset downloaded and extracted!")
print("📁 Location: /content/helmet-dataset")

# Check dataset structure
!ls -la /content/helmet-dataset

### Option B: Use Direct Download (No Kaggle account needed)

In [ ]:
# Alternative: Download from a public source
# Uncomment if Option A doesn't work

# !wget -O helmet-dataset.zip "https://public-dataset-url.com/helmet.zip"
# !unzip -q helmet-dataset.zip -d /content/helmet-dataset
# print("✅ Dataset downloaded!")

## Step 4: Prepare Dataset

In [ ]:
import os
import glob

# Find the data.yaml file
yaml_files = glob.glob('/content/helmet-dataset/**/data.yaml', recursive=True)

if yaml_files:
    data_yaml_path = yaml_files[0]
    print(f"✅ Found data.yaml: {data_yaml_path}")
else:
    # Create data.yaml if not found
    dataset_path = '/content/helmet-dataset'
    
    data_yaml_content = f'''path: {dataset_path}
train: train/images
val: valid/images
test: test/images

nc: 2
names:
  0: with_helmet
  1: without_helmet
'''
    
    data_yaml_path = f'{dataset_path}/data.yaml'
    with open(data_yaml_path, 'w') as f:
        f.write(data_yaml_content)
    
    print(f"✅ Created data.yaml: {data_yaml_path}")

# Display contents
with open(data_yaml_path, 'r') as f:
    print("\n📄 data.yaml contents:")
    print(f.read())

# Check dataset stats
train_images = glob.glob(f'{os.path.dirname(data_yaml_path)}/train/images/*.jpg') + \
               glob.glob(f'{os.path.dirname(data_yaml_path)}/train/images/*.png')
valid_images = glob.glob(f'{os.path.dirname(data_yaml_path)}/valid/images/*.jpg') + \
               glob.glob(f'{os.path.dirname(data_yaml_path)}/valid/images/*.png')

print(f"\n📊 Dataset Statistics:")
print(f"  Training images: {len(train_images)}")
print(f"  Validation images: {len(valid_images)}")

## Step 5: Load YOLOv8 Model

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 small model (fast training, good accuracy)
model = YOLO('yolov8s.pt')

print("✅ YOLOv8s model loaded!")
print("⚙️ Model will be fine-tuned on helmet detection")

## Step 6: Train the Model 🚀

**Training Parameters:**
- Epochs: 50 (quick training, ~30-45 minutes)
- Image size: 640x640
- Batch size: 16
- Optimizer: AdamW

**For better accuracy, increase epochs to 100+**

In [ ]:
import time

start_time = time.time()

# Train the model
results = model.train(
    data=data_yaml_path,
    epochs=50,              # Adjust: 50 for quick, 100+ for best accuracy
    imgsz=640,
    batch=16,               # Reduce to 8 if GPU memory error
    name='helmet-violation-detector',
    patience=15,            # Early stopping
    save=True,
    device=0,               # GPU
    workers=8,
    optimizer='AdamW',
    lr0=0.01,
    weight_decay=0.0005,
    augment=True,
    plots=True,
    verbose=True,
    project='/content/runs',
    exist_ok=True
)

training_time = time.time() - start_time

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)
print(f"⏱️ Training time: {training_time/60:.1f} minutes")

## Step 7: Validate Model Performance

In [ ]:
# Validate the trained model
metrics = model.val()

print("\n📊 Model Performance Metrics:")
print("="*50)
print(f"  mAP50:        {metrics.box.map50:.4f}  (Higher is better, max 1.0)")
print(f"  mAP50-95:     {metrics.box.map:.4f}  (Overall accuracy)")
print(f"  Precision:    {metrics.box.mp:.4f}  (Correct detections)")
print(f"  Recall:       {metrics.box.mr:.4f}  (Found violations)")
print("="*50)

# Interpretation
if metrics.box.map50 > 0.8:
    print("\n🎉 EXCELLENT! Model is highly accurate!")
elif metrics.box.map50 > 0.6:
    print("\n✅ GOOD! Model performs well. Consider training longer for improvement.")
else:
    print("\n⚠️ Model needs improvement. Try training for more epochs.")

## Step 8: Test Predictions (Visual Check)

In [ ]:
# Test on validation images
import glob
from IPython.display import Image, display
import random

# Get random test images
test_images = glob.glob(f'{os.path.dirname(data_yaml_path)}/valid/images/*.jpg')[:5]

if not test_images:
    test_images = glob.glob(f'{os.path.dirname(data_yaml_path)}/test/images/*.jpg')[:5]

print(f"📸 Testing on {len(test_images)} sample images...\n")

# Run predictions
for i, img_path in enumerate(test_images):
    results = model.predict(img_path, conf=0.25, save=True, project='/content/predictions')
    print(f"✅ Processed: {os.path.basename(img_path)}")

# Display first result
prediction_results = glob.glob('/content/predictions/predict*/*.jpg')
if prediction_results:
    print("\n📊 Sample Prediction Result:")
    display(Image(filename=prediction_results[0], width=600))

## Step 9: Save Model to Google Drive ⬇️

In [ ]:
import shutil

# Find best model weights
best_model_path = '/content/runs/detect/helmet-violation-detector/weights/best.pt'

# Copy to Google Drive
shutil.copy(best_model_path, f'{output_dir}/best.pt')

# Also save training results
results_dir = '/content/runs/detect/helmet-violation-detector'
if os.path.exists(f'{results_dir}/results.png'):
    shutil.copy(f'{results_dir}/results.png', f'{output_dir}/training_results.png')
if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    shutil.copy(f'{results_dir}/confusion_matrix.png', f'{output_dir}/confusion_matrix.png')

print("\n" + "="*60)
print("✅ MODEL SAVED TO GOOGLE DRIVE!")
print("="*60)
print(f"\n📁 Location: {output_dir}/")
print("\n📥 FILES SAVED:")
print(f"  ✓ best.pt (Your trained model)")
print(f"  ✓ training_results.png (Training graphs)")
print(f"  ✓ confusion_matrix.png (Performance matrix)")
print("\n🚀 DEPLOYMENT INSTRUCTIONS:")
print("="*60)
print("1. Download 'best.pt' from Google Drive")
print("   Path: MyDrive/traffic-violation-model/best.pt")
print("\n2. Place in your backend folder:")
print("   C:\\Users\\svlak\\New folder (14)\\backend\\best.pt")
print("\n3. Restart backend server:")
print("   cd backend")
print("   .\\start.bat")
print("\n4. Test with traffic images!")
print("   Open: http://localhost:3000")
print("="*60)

# Display file info
import os
model_size = os.path.getsize(f'{output_dir}/best.pt') / (1024*1024)
print(f"\n📊 Model file size: {model_size:.1f} MB")

## Step 10: View Training Results 📈

In [ ]:
# Display training plots
from IPython.display import Image, display

plots_dir = '/content/runs/detect/helmet-violation-detector'

print("📈 Training Performance Graphs:\n")

# Results plot
if os.path.exists(f'{plots_dir}/results.png'):
    print("📊 Loss and Metrics Over Time:")
    display(Image(filename=f'{plots_dir}/results.png', width=900))

# Confusion matrix
if os.path.exists(f'{plots_dir}/confusion_matrix.png'):
    print("\n📊 Confusion Matrix (Prediction Accuracy):")
    display(Image(filename=f'{plots_dir}/confusion_matrix.png', width=600))

# PR curve
if os.path.exists(f'{plots_dir}/PR_curve.png'):
    print("\n📊 Precision-Recall Curve:")
    display(Image(filename=f'{plots_dir}/PR_curve.png', width=600))

## 🎉 Training Complete!

### What You Have Now:
- ✅ **Custom trained model** for helmet violation detection
- ✅ **Model saved** to Google Drive
- ✅ **Performance metrics** showing accuracy
- ✅ **Training visualizations** for analysis

### Next Steps:
1. Download `best.pt` from Google Drive
2. Place it in your backend folder
3. Restart your backend server
4. Upload traffic images and see violations detected!

---

### Want Better Accuracy?
- Increase `epochs` to 100 or 150
- Add more training data
- Use YOLOv8m or YOLOv8l for larger models

**Your traffic violation detection system is ready!** 🚀